In [1]:
!pip install -q imagen-pytorch einops tensorboard satpy cartopy pyresample

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.8/484.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 708.3/708.3 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.6/82.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 127.2 MB/s eta 0:00:00


In [2]:
!pip install -q imagen-pytorch einops tensorboard satpy cartopy pyresample wandb

In [3]:
!pip install -q imagen-pytorch einops tensorboard satpy cartopy pyresample wandb torchmetrics[image] lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.0 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
import os
import glob
import pickle
import random
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from einops import rearrange, repeat
from torch import optim
from torch.utils.tensorboard import SummaryWriter
import logging
import sys
import warnings

# --- CONFIGURATION ---
RUN_NAME = "64_FC_Training_Run"
BASE_DIR = "/kaggle/working/experiment"
DATASET_DIR = "/kaggle/input/cyclone-dataloaders-processed/cyclone_dataloaders"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training Hyperparameters
EPOCHS = 100
BATCH_SIZE = 2  # Keep small for Video training on Kaggle GPU
LR = 3e-4
O_SIZE = 64
N_SIZE = 128
TIMESTEPS = 250
VIDEO_FRAMES = 10

# Create directories
os.makedirs(f"{BASE_DIR}/models/{RUN_NAME}", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs/{RUN_NAME}", exist_ok=True)

print(f"Running on {DEVICE}")

2026-01-17 11:07:44.690638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768648064.918102      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768648064.981364      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768648065.542706      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768648065.542743      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768648065.542746      24 computation_placer.cc:177] computation placer alr

Running on cuda


In [5]:
# --- 1. The Class used to SAVE the data (Required for Pickle Load) ---
class CycloneDataLoader:
    def __init__(self, mode="fc", o_size=64, n_size=128):
        self.mode = mode
        self.img_64_list = []
        self.img_128_list = []
        self.era5_list = []
        self.img_64 = None
        self.img_128 = None
        self.era5 = None
    
    # These methods aren't needed for loading, but good to have for class consistency
    def add_image(self, img_64, img_128, era5): pass
    def finalize(self): pass

# --- 2. The Helper Functions from utils.py ---
def rotate90(x, y, z):
    # Rotates images/videos for augmentation
    # Handle different dimensions for Images vs Videos
    dims = [-2, -1]
    return torch.rot90(x, 1, dims), torch.rot90(y, 1, dims), torch.rot90(z, 1, dims)

def zero_pad(x, t=VIDEO_FRAMES):
    # Pad ERA5 to match video length if needed
    padding = (0, 0, 0, 0, 0, t-1)
    return F.pad(x.unsqueeze(2), padding)

# --- 3. The Video Data Loader (Adapted from v_ModelDataLoader) ---
class v_ModelDataLoader:
    def __init__(self, batch_size, o_size=64, n_size=128, augment=False, test=False, shuffle=True):
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.modality = "img" # Start with image, switch to 'vid' later
        self.augment = augment
        self.test = test
        self.new_data = True
        
        # Buffers
        self.img_cond = torch.empty((0, o_size, o_size), dtype=torch.float32)
        self.img = torch.empty((0, o_size, o_size), dtype=torch.float32)
        self.era5_img = torch.empty((0, 4, o_size, o_size), dtype=torch.float32)
        
        self.vid_cond = torch.empty((0, o_size, o_size), dtype=torch.float32)
        self.vid = torch.empty((0, VIDEO_FRAMES, o_size, o_size), dtype=torch.float32)
        self.era5_vid = torch.empty((0, 4, VIDEO_FRAMES, o_size, o_size), dtype=torch.float32)
        
        self.extremes = torch.empty((0, 2), dtype=torch.float32)

    def switch_to_vid(self):
        self.modality = "vid"
        print("Switched DataLoader to VIDEO mode.")

    def normalize(self, img, max_val=None, min_val=None):
        if max_val is None: max_val = img.max()
        if min_val is None: min_val = img.min()
        # Avoid division by zero
        denom = max_val - min_val
        if denom == 0: denom = 1e-5
        return (img - min_val) / denom

    def add_dataloader(self, cyclone_dataloader):
        # 1. Load Data
        img_o = self.normalize(cyclone_dataloader.img_64) # 64x64 Input
        img_n = cyclone_dataloader.img_128 # Target (Unused for now if 64_FC)
        era5  = cyclone_dataloader.era5 # Context
        
        # Calculate Extremes for normalization tracking
        extreme = torch.tensor([cyclone_dataloader.img_64.max(), cyclone_dataloader.img_64.min()])
        
        # 2. Add to Video Buffers
        # Logic: Chunk data into sequences of length T (10)
        t = VIDEO_FRAMES
        size = era5.shape[0]
        if size % t != 0: size -= size % t # Truncate to fit
        
        for i in range(0, size, t):
            self.extremes = torch.cat((self.extremes, extreme.unsqueeze(0)), 0)
            
            # Target Video: [Batch, T, H, W]
            self.vid = torch.cat((self.vid, img_o[i:i+t].unsqueeze(0)), 0)
            
            # Condition: First frame of ERA5 (ch 0 is prev_sat)
            # We use ERA5 channel 0 as "Previous Frame Condition"
            vid_cond = self.normalize(era5[i:i+1, 0, :, :], extreme[0], extreme[1])
            self.vid_cond = torch.cat((self.vid_cond, vid_cond), 0)
            
            # ERA5 Context Video: [Batch, 4, T, H, W]
            # Rearrange ERA5 chunk to [1, 4, T, H, W]
            era5_chunk = era5[i:i+t].permute(1, 0, 2, 3).unsqueeze(0) 
            self.era5_vid = torch.cat((self.era5_vid, era5_chunk), 0)

        self.new_data = True

    def create_batches(self, batch_size):
        # Determine current dataset size based on modality
        if self.modality == "vid":
            size = self.vid.shape[0]
        else:
            size = self.img.shape[0] # Not implemented in this simplified version
            
        if size == 0: return 
        
        # Drop last incomplete batch
        if size % batch_size != 0:
            size -= size % batch_size
            
        idx = torch.arange(size)
        if self.shuffle: idx = torch.randperm(size)
        self.random_idx = idx.reshape(-1, batch_size)

    def vid_to3channel(self, x):
        # Expand 1-channel grayscale to 3-channel RGB for Imagen
        # Input: [Batch, T, H, W] or [Batch, H, W]
        if len(x.shape) == 3: # [B, H, W] -> [B, 3, H, W]
            return x.unsqueeze(1).expand(-1, 3, -1, -1)
        if len(x.shape) == 4: # [B, T, H, W] -> [B, 3, T, H, W]
            return x.unsqueeze(1).expand(-1, 3, -1, -1, -1)
        return x

    def __len__(self):
        if self.new_data: self.create_batches(self.batch_size)
        self.new_data = False
        return self.random_idx.shape[0]

    def __iter__(self):
        self.batch_idx = 0
        return self

    def __next__(self):
        if self.batch_idx < self.random_idx.shape[0]:
            batch_indices = self.random_idx[self.batch_idx]
            self.batch_idx += 1
            
            # Retrieve Batch
            if self.modality == "vid":
                # Returns: 
                # 1. Condition Video (Prev Frame) [B, 3, H, W] (unsqueezed to pretend it's video? No, Imagen expects image cond)
                # 2. Target Video [B, 3, T, H, W]
                # 3. Continuous Embeds (ERA5) [B, 4, T, H, W]
                
                cond = self.vid_to3channel(self.vid_cond[batch_indices]).float().cuda()
                target = self.vid_to3channel(self.vid[batch_indices]).float().cuda()
                context = self.era5_vid[batch_indices].float().cuda()
                
                # Reshape condition to be [B, 3, 1, H, W] for Imagen video conditioning
                cond = cond.unsqueeze(2) 
                
                return cond, target, context
        else:
            raise StopIteration

In [6]:
# --- Initialize Loaders ---
# FIX: Removed 'mode="fc"' because the simplified class defaults to forecasting logic
train_loader = v_ModelDataLoader(BATCH_SIZE, O_SIZE, N_SIZE, augment=False)
test_loader  = v_ModelDataLoader(BATCH_SIZE, O_SIZE, N_SIZE, augment=False, test=True)

# Find Files
dat_files = glob.glob(f"{DATASET_DIR}/*.dat")
if not dat_files:
    # Fallback search if the path structure is slightly different
    dat_files = glob.glob(f"/kaggle/input/**/*.dat", recursive=True)

if not dat_files:
    raise FileNotFoundError(f"No .dat files found in {DATASET_DIR}. Please check the Input path.")

print(f"Found {len(dat_files)} cyclone files.")

# Random Split (80/20)
random.seed(42) # Set seed for reproducibility
random.shuffle(dat_files)
split_idx = int(len(dat_files) * 0.8)
train_files = dat_files[:split_idx]
test_files = dat_files[split_idx:]

print(f"Loading {len(train_files)} files for Training...")
for fn in tqdm(train_files, desc="Train Files"):
    try:
        with open(fn, "rb") as f:
            data = pickle.load(f)
            train_loader.add_dataloader(data)
    except Exception as e: print(f"Error loading {fn}: {e}")

print(f"Loading {len(test_files)} files for Testing...")
for fn in tqdm(test_files, desc="Test Files"):
    try:
        with open(fn, "rb") as f:
            data = pickle.load(f)
            test_loader.add_dataloader(data)
    except Exception as e: print(f"Error loading {fn}: {e}")

# Prepare Data (Switch to video mode and batch)
train_loader.switch_to_vid()
train_loader.create_batches(BATCH_SIZE)
test_loader.switch_to_vid()
test_loader.create_batches(BATCH_SIZE)

print(f"Train Batches: {len(train_loader)}")
print(f"Test Batches:  {len(test_loader)}")

Found 5 cyclone files.
Loading 4 files for Training...


Train Files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading 1 files for Testing...


Test Files:   0%|          | 0/1 [00:00<?, ?it/s]

Switched DataLoader to VIDEO mode.
Switched DataLoader to VIDEO mode.
Train Batches: 29
Test Batches:  4


In [7]:
from imagen_pytorch import Unet3D, Imagen
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import gc
import wandb
from kaggle_secrets import UserSecretsClient

# --- 0. WANDB SETUP ---
try:
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_key)
    print("✅ Logged into WandB using Kaggle Secret.")
except:
    print("⚠️ Kaggle Secret 'wandb_api_key' not found. Trying anonymous mode.")
    wandb.login(anonymous='allow')

# Initialize Run
run = wandb.init(
    project="Cyclone-Video-Diffusion",
    name=RUN_NAME,
    config={
        "epochs": EPOCHS,
        "batch_size": 1,
        "grad_accum_steps": 8,
        "learning_rate": 1e-4,
        "model_dim": 128,
        "timesteps": TIMESTEPS,
        "image_size": O_SIZE,
        "video_frames": VIDEO_FRAMES,
        "projector": "Convolutional"
    }
)

# --- 1. FORCE MEMORY CLEAR ---
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except:
        pass
clear_gpu()

# --- 2. Conditioning Projector (Conv) ---
class ConvERA5Projector(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        # Input: [Batch, 4, 10, 64, 64] -> Downsample -> [Batch, 2560]
        self.pool = nn.AdaptiveAvgPool3d((None, 8, 8)) 
        self.flatten_dim = 4 * VIDEO_FRAMES * 8 * 8
        
        self.net = nn.Sequential(
            nn.Linear(self.flatten_dim, output_dim),
            nn.SiLU(),
            nn.Linear(output_dim, output_dim),
            nn.LayerNorm(output_dim)
        )
    
    def forward(self, x):
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)
        return self.net(x).unsqueeze(1)

MODEL_COND_DIM = 1024 
projector = ConvERA5Projector(output_dim=MODEL_COND_DIM).to(DEVICE)

# --- 3. Model Architecture ---
unet1 = Unet3D(
    dim = 128, 
    cond_dim = MODEL_COND_DIM,
    dim_mults = (1, 2, 4, 8),
    num_resnet_blocks = 1,      
    layer_attns = (False, False, False, True), 
    channels = 3,          
    cond_images_channels = 3 
)

imagen = Imagen(
    unets = [unet1],
    image_sizes = (O_SIZE),
    timesteps = TIMESTEPS,
    cond_drop_prob = 0.1,
    condition_on_text = True,      
    text_embed_dim = MODEL_COND_DIM,
    auto_normalize_img = False
).to(DEVICE)

# --- 4. Optimizer & Scaler ---
params = list(imagen.parameters()) + list(projector.parameters())
optimizer = optim.Adam(params, lr=1e-4)
scaler = GradScaler() 

# --- 5. Training Loop ---
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8 

print(f"🚀 Starting Training: dim=128 | Batch={BATCH_SIZE} | Accum={GRAD_ACCUM_STEPS}")

train_loader.create_batches(BATCH_SIZE)

for epoch in range(EPOCHS):
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    optimizer.zero_grad()
    current_accum_step = 0
    
    for i, (vid_cond, target_vid, era5_context) in enumerate(pbar):
        # Move to GPU
        cond_img = vid_cond.squeeze(2).float().to(DEVICE)
        target_vid = target_vid.float().to(DEVICE)
        era5_context = era5_context.float().to(DEVICE)
        
        # Forward Pass (FP16)
        with autocast():
            cond_embeds = projector(era5_context)
            
            loss = imagen(
                target_vid, 
                cond_images = cond_img,
                text_embeds = cond_embeds,
                unet_number = 1
            )
            loss = loss / GRAD_ACCUM_STEPS
        
        # Backward (Scaled)
        scaler.scale(loss).backward()
        
        current_accum_step += 1
        
        # Update Weights
        if current_accum_step % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            current_accum_step = 0
            
            # Clean variables to prevent OOM
            del cond_embeds, cond_img, target_vid
            
        # Logging
        actual_loss = loss.item() * GRAD_ACCUM_STEPS
        total_loss += actual_loss
        pbar.set_postfix(MSE=actual_loss)
        
        # Log to WandB Live
        global_step = epoch * len(train_loader) + i
        wandb.log({
            "train/mse_loss": actual_loss,
            "train/epoch": epoch + 1,
            "train/global_step": global_step,
            "system/learning_rate": optimizer.param_groups[0]['lr']
        })

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.5f}")
    wandb.log({"train/epoch_avg_loss": avg_loss})
    
    # Save Checkpoint
    if (epoch + 1) % 5 == 0 or epoch == 0:
        save_path_model = f"{BASE_DIR}/models/{RUN_NAME}/ckpt_epoch_{epoch+1}.pt"
        save_path_proj = f"{BASE_DIR}/models/{RUN_NAME}/proj_epoch_{epoch+1}.pt"
        torch.save(imagen.state_dict(), save_path_model)
        torch.save(projector.state_dict(), save_path_proj)
        
        # Optional: Save model to WandB Artifacts (Good for versioning)
        # artifact = wandb.Artifact(f'model-epoch-{epoch+1}', type='model')
        # artifact.add_file(save_path_model)
        # run.log_artifact(artifact)
        
        print(f"Saved checkpoint: {save_path_model}")

print("Training Complete!")
wandb.finish()

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

⚠️ Kaggle Secret 'wandb_api_key' not found. Trying anonymous mode.


wandb: setting up run fpsa7id8
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260117_110814-fpsa7id8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run 64_FC_Training_Run
wandb: ⭐️ View project at https://wandb.ai/vedanggg-mit-manipal/Cyclone-Video-Diffusion?apiKey=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
wandb: 🚀 View run at https://wandb.ai/vedanggg-mit-manipal/Cyclone-Video-Diffusion/runs/fpsa7id8?apiKey=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
wandb: WARNING Do NOT share these links with anyone. They can be used to claim your runs.


🚀 Starting Training: dim=128 | Batch=1 | Accum=8


...

Epoch 100/100:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 100 Average Loss: 0.12391


wandb: updating run metadata


Saved checkpoint: /kaggle/working/experiment/models/64_FC_Training_Run/ckpt_epoch_100.pt
Training Complete!


wandb: 
wandb: Run history:
wandb: system/learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:          train/epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▇▇████
wandb: train/epoch_avg_loss ████▇▆▆▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    train/global_step ▁▁▁▁▁▁▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇█
wandb:       train/mse_loss ██▁▇▃▃▄▃▂▃▃▂▃▂▂▄▂▂▄▁▃▂▁▂▃▂▃▂▂▁▂▂▂▁▂▃▁▁▂▂
wandb: 
wandb: Run summary:
wandb: system/learning_rate 0.0001
wandb:          train/epoch 100
wandb: train/epoch_avg_loss 0.12391
wandb:    train/global_step 5799
wandb:       train/mse_loss 0.10762
wandb: 
wandb: 🚀 View run 64_FC_Training_Run at: https://wandb.ai/vedanggg-mit-manipal/Cyclone-Video-Diffusion/runs/fpsa7id8?apiKey=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
wandb: ⭐️ View project at: https://wandb.ai/vedanggg-mit-manipal/Cyclone-Video-Diffusion?apiKey=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
wandb: Synced 5 W&B file(